# 🧪 Exploring Self-Supervised Learning with DINO for Image Classification

## 🎯 1. Objective
Use the DINO pretrained ViT (Vision Transformer) model to extract meaningful image features without labels.

* Perform linear evaluation by training a simple linear classifier on top of frozen features.
* Compare results with a supervised baseline on a small labeled dataset.
* Gain insights into the power of self-supervised representations for downstream tasks.

## 📦 2. Python Dependencies

To run this notebook, make sure you have the following packages installed:

```bash
pip install torch torchvision matplotlib scikit-learn seaborn timm
```

⚠️ If you're using Google Colab or a similar environment, most of these libraries are pre-installed.

## ⚙️ 3. Import Dependencies

In [1]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 🗂️ 3. Load and Preprocess Dataset

* Download the CIFAR10 dataset with a few classes
* Resize images to 224×224 to match DINI pretrained input requirements.
* Create a small dataset with only 200 images per class. 


In [2]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 🧪 Load CIFAR-10
full_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

# 🎯 Keep only 3 classes
selected_classes = [0, 1, 2]
selected_indices = [i for i, (_, label) in enumerate(full_dataset) if label in selected_classes]

# 🎯 Limit to 200 samples per class
class_counts = {c: 0 for c in selected_classes}
limited_indices = []

for idx in selected_indices:
    label = full_dataset[idx][1]
    if class_counts[label] < 200:
        limited_indices.append(idx)
        class_counts[label] += 1
    if all(c >= 200 for c in class_counts.values()):
        break

reduced_dataset = Subset(full_dataset, limited_indices)


Files already downloaded and verified


## ⬇️ 4. Load DINO Pretrained ViT Model

In [3]:
# Some functions are not available for mps
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 

# Load DINO pretrained ViT backend from Facebook Research
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

model.eval().to(device)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /home/torch/hub/main.zip
/home/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /home/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:08<00:00, 10.1MB/s]


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

## 🧬 5. Extract Features for Dataset

In [4]:
def extract_features(dataset, model, device, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features = []
    labels = []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            feats = model(images)
            features.append(feats.cpu())
            labels.extend(targets.numpy())
    features = torch.cat(features).numpy()
    return features, np.array(labels)

features, labels = extract_features(reduced_dataset, model, device)


## 🏋️‍♂️ 6. Train Logistic Regression on Features

In [5]:
torch.manual_seed(22)

train_split, val_split = torch.utils.data.random_split(range(len(features)), [0.8, 0.2])

X_train, y_train = features[train_split], labels[train_split]
X_val, y_val = features[val_split], labels[val_split] 

clf = LogisticRegression(max_iter=1000, solver='lbfgs')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_val)

acc = accuracy_score(y_val, y_pred)
print(f'Linear classifier accuracy on DINO features: {acc:.4f}')


Linear classifier accuracy on DINO features: 0.9917


## 🆚 8. Baseline: Train CNN on Same Dataset  

In [6]:
from torchvision import transforms, models

# Fine tune a resnet18 pretrained on ImageNet
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Modify the classifier head
num_classes = len(selected_classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Split into train/test

train_dataset = torch.utils.data.Subset(reduced_dataset, train_split)
val_dataset = torch.utils.data.Subset(reduced_dataset, val_split)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

def train_model(model, loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")
    print("Training complete.")

train_model(model, train_loader, criterion, optimizer)


Epoch 1/5, Loss: 1.0034
Epoch 2/5, Loss: 0.6838
Epoch 3/5, Loss: 0.5247
Epoch 4/5, Loss: 0.4212
Epoch 5/5, Loss: 0.3600
Training complete.


In [7]:
def evaluate_model(model, loader):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = correct / total
    print(f"Test Accuracy: {accuracy:.4f}")


evaluate_model(model, val_loader)


Test Accuracy: 0.9417


## 📊 9. Compare Results

Compare accuracy and training times between:

* Linear classification on DINO features 

* Training CNN

## 🤔 9. Discussion & Questions
1. How does self-supervised pretraining affect classification performance?

   Self-supervised learning use unlabeled input data, 
3. When would you prefer apply a linear classifier over full CNN training?
4. What are the limitations of self-supervised models like DINO?

CNN would get better solution over linear classifier.